# SENTINEL — EDA
Exploration only. **Training lives in `src/train.py`** (`python -m src.train`), so notebook and production code can't drift apart.

Run from the repo root. Needs `data/creditcard.csv` (real) or run `python scripts/make_synthetic.py` and point `PATH` at the synthetic file.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.train import load_dataset, time_split
from src.features import make_features

PATH = '../data/creditcard.csv'   # or '../data/synthetic_creditcard.csv'
df = load_dataset(PATH)
print(df.shape, 'fraud rate = %.4f%%' % (100 * df.Class.mean()))
df.head()

## Class balance and time-ordered split
The model is trained on the earliest 60 %, calibrated on the next 20 % and tested once on the latest 20 %.

In [ ]:
train, val, test = time_split(df, 0.6, 0.2)
pd.DataFrame({'rows': [len(train), len(val), len(test)], 'frauds': [train.Class.sum(), val.Class.sum(), test.Class.sum()]}, index=['train', 'val', 'test'])

## Amount and time-of-day
With few frauds, any apparent pattern here is noisy — check the confidence intervals in `docs/MODEL_CARD.md` before believing a feature helps.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for c, g in df.groupby('Class'):
    ax[0].hist(np.log1p(g.Amount), bins=40, alpha=.6, density=True, label=f'Class {c}')
ax[0].set_xlabel('log1p(Amount)'); ax[0].legend()
hour = (df.Time % 86400) // 3600
ax[1].plot(df.groupby(hour).Class.mean()); ax[1].set_xlabel('hour of day'); ax[1].set_ylabel('fraud rate')
plt.tight_layout()

## Feature separation (|mean difference| in std units)

In [ ]:
X = make_features(df)
sep = ((X[df.Class == 1].mean() - X[df.Class == 0].mean()).abs() / X.std()).sort_values(ascending=False)
sep.head(12).plot.barh(figsize=(5, 4)).invert_yaxis()

## Training and results
```bash
python -m src.train            # real data
python -m src.train --synthetic
```
Then see `model/metrics.json`, `docs/MODEL_CARD.md`, `docs/figures/`.